## Score trees based on monophyleticity

> Leaves are named with group code, followed by an underscore.
> We check every internal node, see if all its leaves belong to the same group.
> If yes, such internode is monophyletic.
> The script reports the number of monophyletic internal nodes, the number of all internal nodes, and the fraction of monophyletic internal nodes.

In [1]:
from Bio import Phylo
import re


def score_monophyly(filename):
    """
    Calculate the fraction of internal nodes whose descendant leaves all
    belong to the same group.

    Leaf names must follow this format:
        ABCD_[unique_name]

    Parameters
    ----------
    filename : str
        Path to the input Newick file.

    Returns
    -------
    tuple
        (monophyletic_internal_nodes, all_internal_nodes, fraction)
    """
    tree = Phylo.read(filename, "newick")
    group_pattern = re.compile(r"^([A-Za-z0-9\-]+)_")

    def get_group(leaf):
        if not leaf.name:
            raise ValueError("Encountered a leaf without a name.")

        match = group_pattern.match(leaf.name)
        if not match:
            raise ValueError(
                f"Invalid leaf name {leaf.name!r}: expected a four-letter "
                "group code followed by an underscore."
            )

        return match.group(1)

    internal_nodes = tree.get_nonterminals()
    monophyletic_nodes = 0

    for node in internal_nodes:
        descendant_groups = {
            get_group(leaf) for leaf in node.get_terminals()
        }

        if len(descendant_groups) == 1:
            monophyletic_nodes += 1

    total_nodes = len(internal_nodes)
    fraction = monophyletic_nodes / total_nodes if total_nodes else 0.0

    #print("-" * 40)
    #print(f"Filename: {filename}")
    print(f"Monophyletic internal nodes: {monophyletic_nodes}")
    print(f"All internal nodes: {total_nodes}")
    print(f"Fraction of monophyletic internal nodes: {fraction:.6f}")
    print("-" * 40)

    # return monophyletic_nodes, total_nodes, fraction

### Flaviviridae NS5 (RdRp)

In [2]:
print("Mifsud")
score_monophyly("nwk/NS5_aa.nwk")
print("ProstT5")
score_monophyly("nwk/NS5_prostt5.nwk")
print("ESM3Di")
score_monophyly("nwk/NS5_esm3di.nwk")


Mifsud
Monophyletic internal nodes: 437
All internal nodes: 460
Fraction of monophyletic internal nodes: 0.950000
----------------------------------------
ProstT5
Monophyletic internal nodes: 434
All internal nodes: 460
Fraction of monophyletic internal nodes: 0.943478
----------------------------------------
ESM3Di
Monophyletic internal nodes: 435
All internal nodes: 460
Fraction of monophyletic internal nodes: 0.945652
----------------------------------------


### Flaviviridae E/E1/E2 mixture

In [3]:
print("AlphaFold - FoldTree")
score_monophyly("nwk/E_af2.nwk")
print("ProstT5 - FoldTree")
score_monophyly("nwk/E_prostt5.nwk")
print("ESM3Di - FoldTree")
score_monophyly("nwk/E_esm3di.nwk")
print("AA - MAFFT - IQ-TREE ModelFinder")
score_monophyly("nwk/E_aa.nwk")

AlphaFold - FoldTree
Monophyletic internal nodes: 490
All internal nodes: 625
Fraction of monophyletic internal nodes: 0.784000
----------------------------------------
ProstT5 - FoldTree
Monophyletic internal nodes: 521
All internal nodes: 625
Fraction of monophyletic internal nodes: 0.833600
----------------------------------------
ESM3Di - FoldTree
Monophyletic internal nodes: 556
All internal nodes: 625
Fraction of monophyletic internal nodes: 0.889600
----------------------------------------
AA - MAFFT - IQ-TREE ModelFinder
Monophyletic internal nodes: 541
All internal nodes: 625
Fraction of monophyletic internal nodes: 0.865600
----------------------------------------


---

### Filoviridae (Ebolavirus)

In [4]:
print("GP - ESM3Di - ProstT5")
score_monophyly("/work/FAC/FBM/DBC/cdessim2/default/dkim5/projects/viral/ebola/nwk/gp_foldtree_prostt5.renamed.nwk")
print("GP - ESM3Di - FoldTree")
score_monophyly("/work/FAC/FBM/DBC/cdessim2/default/dkim5/projects/viral/ebola/nwk/gp_foldtree_esm3di.renamed.nwk")
print("NP - ESM3Di - ProstT5")
score_monophyly("/work/FAC/FBM/DBC/cdessim2/default/dkim5/projects/viral/ebola/nwk/np_foldtree_prostt5.renamed.nwk")
print("NP - ESM3Di - FoldTree")
score_monophyly("/work/FAC/FBM/DBC/cdessim2/default/dkim5/projects/viral/ebola/nwk/np_foldtree_esm3di.renamed.nwk")
print("L - ESM3Di - ProstT5")
score_monophyly("/work/FAC/FBM/DBC/cdessim2/default/dkim5/projects/viral/ebola/nwk/l_foldtree_prostt5.renamed.nwk")
print("L - ESM3Di - FoldTree")
score_monophyly("/work/FAC/FBM/DBC/cdessim2/default/dkim5/projects/viral/ebola/nwk/l_foldtree_esm3di.renamed.nwk")

GP - ESM3Di - ProstT5
Monophyletic internal nodes: 2810
All internal nodes: 2868
Fraction of monophyletic internal nodes: 0.979777
----------------------------------------
GP - ESM3Di - FoldTree
Monophyletic internal nodes: 2852
All internal nodes: 2868
Fraction of monophyletic internal nodes: 0.994421
----------------------------------------
NP - ESM3Di - ProstT5
Monophyletic internal nodes: 3369
All internal nodes: 3375
Fraction of monophyletic internal nodes: 0.998222
----------------------------------------
NP - ESM3Di - FoldTree
Monophyletic internal nodes: 3370
All internal nodes: 3375
Fraction of monophyletic internal nodes: 0.998519
----------------------------------------
L - ESM3Di - ProstT5
Monophyletic internal nodes: 3596
All internal nodes: 3601
Fraction of monophyletic internal nodes: 0.998611
----------------------------------------
L - ESM3Di - FoldTree
Monophyletic internal nodes: 3596
All internal nodes: 3601
Fraction of monophyletic internal nodes: 0.998611
--------

---

## Plot trees

### Flaviviridae - based on six-letter codes (e.g. E1HPHV_...)

In [8]:
# -------------------- CHANGE THESE FLAGS --------------------
LABEL_MODE = "six_letter"  # "scientific" or "six_letter"
TREE_LAYOUT = "circular"    # "radial" (legends) or "circular" (arcs)

# Radial spacing only. These settings do not change branch lengths.
RADIAL_SPACING = "balanced"  # "balanced" or "original" (Toytree layout)
CLADE_SIZE_POWER = 0.2       # 0: equal space per subclade; 1: equal per tip
# Smaller values compress densely sampled subclades more strongly.
# ---------------------------------------------------------------

In [9]:
"""Plot a Newick tree with hierarchical colors using Toytree.

Install: pip install toytree toyplot numpy
Run: python plot_annotated_tree.py input.nwk -o tree.svg
Tested with toytree 3.0.11 and toyplot 2.1.0.

Scientific names: Orthomarburgvirus-marburgense_XJQ59011
  Orthomarburgvirus determines the hue; marburgense determines the shade.
Six-letter codes: E1HPHV_Recombinant...
  E1 determines the hue; HP and then HV determine related shades.
  E0 is displayed as E; circular arc labels contain only HPHV.

Radial uses a weighted equal-angle layout by default. Clade membership is
evaluated using the input Newick root. No subtree swaps are performed.
Circular arcs mark maximal pure subtrees; a fragmented group may have several
arcs. Set strict_monophyly=True to require all members in a single subtree.
Consecutive fragments of the same subclade share one midpoint label, including
across the circular seam. Arc gaps still show the separate pure subtrees.
"""

from collections import Counter
import colorsys
from html import escape
from pathlib import Path
import re

import numpy as np
import toyplot
import toyplot.marker
import toyplot.svg
import toytree


CODE_COLORS = {"E0": "#7B4AB5", "E1": "#2479B5", "E2": "#D97720"}
MIXED_COLOR = "#A0A0A0"


def _mode(label_mode):
    mode = LABEL_MODE if label_mode is None else label_mode
    if mode not in ("scientific", "six_letter"):
        raise ValueError('LABEL_MODE must be "scientific" or "six_letter".')
    return mode


def _parse_tip(name, mode):
    if mode == "scientific":
        higher, sep, lower = name.split("_", 1)[0].partition("-")
        if sep and higher and lower:
            return higher, lower
        raise ValueError(f"Expected Genus-species_accession; got {name!r}.")
    match = re.match(r"^([A-Za-z0-9]{6})(?:_|$)", name)
    if match:
        code = match.group(1)
        return code[:2], code[2:4], code[4:6]
    raise ValueError(f"Expected a six-letter prefix such as E1HPHV_; got {name!r}.")


def _hex(rgb):
    return "#" + "".join(f"{round(c * 255):02x}" for c in rgb)


def _palette(keys, mode, higher_colors=None):
    groups = sorted({k[0] for k in keys})
    if higher_colors is None:
        if mode == "six_letter":
            higher_colors = dict(CODE_COLORS)
        else:
            higher_colors = {
                g: _hex(colorsys.hls_to_rgb((0.58 + i / len(groups)) % 1, .46, .68))
                for i, g in enumerate(groups)
            }
    higher_colors = dict(higher_colors)
    missing = set(groups) - higher_colors.keys()
    if missing:
        raise ValueError(f"Supply higher_colors for: {sorted(missing)}")
    colors = {}
    for group in groups:
        base = toyplot.color.css(higher_colors[group])
        h, light, s = colorsys.rgb_to_hls(*(float(base[c]) for c in ("r", "g", "b")))
        if mode == "scientific":
            members = sorted(k for k in keys if k[0] == group)
            levels = [light] if len(members) == 1 else np.linspace(.30, .72, len(members))
            for key, level in zip(members, levels):
                colors[key] = _hex(colorsys.hls_to_rgb(h, float(level), s))
        else:
            higher = sorted({k[1] for k in keys if k[0] == group})
            centers = [light] if len(higher) == 1 else np.linspace(.36, .68, len(higher))
            spread = min(.08, .11 / len(higher))
            for parent, center in zip(higher, centers):
                members = sorted(k for k in keys if k[:2] == (group, parent))
                levels = [center] if len(members) == 1 else np.linspace(
                    max(.20, center - spread), min(.82, center + spread), len(members)
                )
                for key, level in zip(members, levels):
                    colors[key] = _hex(colorsys.hls_to_rgb(h, float(level), s))
    return higher_colors, colors


def make_clade_colors(tree_files, *, label_mode=None, higher_colors=None):
    """Build a shared palette before plotting multiple Newick files.

    Return (higher_colors, subclade_colors); pass both to every plot_tree call
    to keep colors consistent when different files contain different taxa.
    Subclade keys are (genus, species) or (E1, HP, HV), depending on mode.
    """
    mode = _mode(label_mode)
    keys = set()
    for filename in tree_files:
        tree = toytree.tree(str(filename))
        keys.update(_parse_tip(name, mode) for name in tree.get_tip_labels())
    if not keys:
        raise ValueError("No tips found in the supplied files.")
    return _palette(keys, mode, higher_colors)


def _summarize(tree, mode):
    higher, lower = [None] * tree.nnodes, [None] * tree.nnodes
    bounds = np.zeros((tree.nnodes, 2), dtype=int)
    counts = np.zeros(tree.nnodes, dtype=int)
    for node in tree:
        i = node.idx
        if node.is_leaf():
            lower[i] = _parse_tip(node.name, mode)
            higher[i] = lower[i][0]
            bounds[i], counts[i] = (i, i), 1
        else:
            children = [c.idx for c in node.children]
            for groups in (higher, lower):
                first = groups[children[0]]
                if first is not None and all(groups[j] == first for j in children):
                    groups[i] = first
            bounds[i] = min(bounds[j, 0] for j in children), max(bounds[j, 1] for j in children)
            counts[i] = sum(counts[j] for j in children)
    return higher, lower, bounds, counts


def _name(key, mode):
    return " ".join(key) if mode == "scientific" else "".join(key[1:])


def _balanced_radial_coordinates(tree, tip_keys, power):
    """Allocate angular sectors by subclade size raised to ``power``.

    Each tip in a subclade of n tips has weight n**(power - 1), so that
    subclade has total weight n**power. Repeated taxa therefore need not
    consume most of the available angle. Keys represent label groups, not
    sequence identity. Nonmonophyletic groups remain in separate positions.

    Sectors follow the input child order. Every edge keeps its length, and
    every tip remains present. Coordinates are computed iteratively.
    """
    if not np.isfinite(power) or not 0 <= power <= 1:
        raise ValueError("clade_size_power must be between 0 and 1.")
    totals = Counter(tip_keys)
    weights = np.zeros(tree.nnodes)
    weights[:tree.ntips] = [totals[key] ** (power - 1) for key in tip_keys]
    for node in tree:
        if not node.is_leaf():
            weights[node.idx] = sum(weights[c.idx] for c in node.children)
    coordinates = np.zeros((tree.nnodes, 2))
    sectors = np.zeros((tree.nnodes, 2))
    sectors[tree.treenode.idx] = 0, 2 * np.pi
    for parent in tree.traverse("preorder"):
        start, end = sectors[parent.idx]
        cursor = start
        for child in parent.children:
            width = (end - start) * weights[child.idx] / weights[parent.idx]
            sectors[child.idx] = cursor, cursor + width
            midpoint = cursor + width / 2
            coordinates[child.idx] = coordinates[parent.idx] + child.dist * np.array(
                [np.sin(midpoint), -np.cos(midpoint)]
            )
            cursor += width
    return coordinates, sectors


def _label(text, mode):
    return f"<i>{escape(text)}</i>" if mode == "scientific" else escape(text)


def _display_group(group):
    return "E" if group == "E0" else group


def _legend_rows(keys, mode, higher_colors, colors):
    rows = [("Higher clades" if mode == "scientific" else "Highest clades", None)]
    rows += [(_label(g if mode == "scientific" else _display_group(g), mode), higher_colors[g])
             for g in sorted({k[0] for k in keys})]
    rows += [("Subclades", None)]
    for key in sorted(keys):
        text = _name(key, mode)
        if mode == "six_letter":
            text = f"{_display_group(key[0])}: {text}"
        rows.append((_label(text, mode), colors[key]))
    return rows


def _draw_legends(canvas, rows, size, column_width, rows_per_column):
    for column, start in enumerate(range(0, len(rows), rows_per_column)):
        entries = []
        for label, color in rows[start:start + rows_per_column]:
            marker = toyplot.marker.create(
                shape="o", size=8,
                mstyle={"fill": color or "none", "stroke": "none"},
            )
            entries.append((label if color else f"<b>{label}</b>", marker))
        x = size + column * column_width
        legend = canvas.legend(entries, bounds=(x, x + column_width, 40, 40 + 23 * len(entries)))
        legend.cells.column[0].width = 24
        legend.cells.column[1].lstyle = {"font-size": "12px"}


def _arc_label_blocks(fragments, tip_keys):
    """Group neighboring fragments for labeling without merging their arcs.

    Compare full clade keys, not displayed names or colors. An intervening tip
    from another group always breaks a block, even if it has no displayed arc.
    Blocks may cross the 0/360-degree seam; their end angle is then unwrapped.
    """
    blocks = []
    for fragment in sorted(fragments, key=lambda f: f["lo"]):
        key = fragment["key"]
        if (blocks and blocks[-1]["key"] == key and
                all(k == key for k in tip_keys[blocks[-1]["hi"] + 1:fragment["lo"]])):
            blocks[-1].update(hi=fragment["hi"], end=fragment["end"])
            blocks[-1]["fragments"] += 1
        else:
            blocks.append(dict(fragment, fragments=1))
    if len(blocks) > 1:
        first, last = blocks[0], blocks[-1]
        key = first["key"]
        seam_keys = tip_keys[last["hi"] + 1:] + tip_keys[:first["lo"]]
        if last["key"] == key and all(k == key for k in seam_keys):
            wrapped = dict(last, end=first["end"] + 2 * np.pi,
                           hi=first["hi"] + len(tip_keys),
                           fragments=last["fragments"] + first["fragments"])
            blocks = blocks[1:-1] + [wrapped]
    return blocks


def _arcs(axes, tree, mark, lower, bounds, counts, colors, mode,
          plot_span, domain_scale, arc_width, min_clade_size, strict_monophyly):
    center = np.asarray(mark.ntable[tree.treenode.idx])
    # Circular layout's ttable retains the intended angle even for zero-length tips.
    tip_xy = np.asarray(mark.ttable) - center
    angles = np.unwrap(np.arctan2(tip_xy[:, 1], tip_xy[:, 0]))
    radius = np.max(np.linalg.norm(np.asarray(mark.ntable) - center, axis=1))
    if not np.isfinite(radius) or radius <= 0:
        raise ValueError("The tree has no positive drawing radius.")
    step, ring = 2 * np.pi / tree.ntips, 1.045
    totals, outside_angles = Counter(lower[:tree.ntips]), []
    stats = {"inside": 0, "outside": 0}
    fragments = []
    for node in tree:
        i, key = node.idx, lower[node.idx]
        if key is None or counts[i] < min_clade_size:
            continue
        if node.up is not None and lower[node.up.idx] == key:
            continue
        if strict_monophyly and counts[i] != totals[key]:
            continue
        lo, hi = bounds[i]
        start, end = angles[lo] - .4 * step, angles[hi] + .4 * step
        if counts[i] == tree.ntips:
            start = angles[0] - .5 * step
            end = start + 2 * np.pi
        theta = np.linspace(start, end, max(8, int((end - start) * 100)))
        color = colors[key]
        axes.plot(*(center[:, None] + radius * ring * np.array([np.cos(theta), np.sin(theta)])),
                  color=color, stroke_width=arc_width, style={"stroke-linecap": "butt"})
        fragments.append(dict(key=key, lo=int(lo), hi=int(hi),
                              start=float(start), end=float(end)))

    # Label the complete consecutive block, while keeping the original arc
    # fragments separate so adjacency is not mistaken for monophyly.
    blocks = _arc_label_blocks(fragments, lower[:tree.ntips])
    for block in blocks:
        key, start, end = block["key"], block["start"], block["end"]
        color = colors[key]
        midpoint = (start + end) / 2
        direction = np.array([np.cos(midpoint), np.sin(midpoint)])
        text = _name(key, mode)
        available_px = (end - start) * plot_span * ring / (2 * domain_scale)
        if available_px >= len(text) * 6.2 + 12:
            stats["inside"] += 1
            label_radius = ring - .045
        else:
            stats["outside"] += 1
            lane = min(2, sum(abs(np.angle(np.exp(1j * (midpoint - a)))) < .20
                              for a in outside_angles))
            outside_angles.append(midpoint)
            leader_end = 1.12 + .075 * lane
            label_radius = leader_end + .025
            xy = center[:, None] + radius * direction[:, None] * np.array([ring + .008, leader_end])
            axes.plot(*xy, color=color, stroke_width=max(1, arc_width / 5))
        rotation = (np.degrees(midpoint) + 90) % 360
        if 90 < rotation < 270:
            rotation -= 180
        xy = center + radius * label_radius * direction
        axes.text(*xy, _label(text, mode), angle=rotation,
                  style={"fill": color, "font-size": "11px", "text-anchor": "middle"})
    mark.arc_label_stats = stats
    mark.arc_fragments = fragments
    mark.arc_label_blocks = blocks
    return center, radius * domain_scale


def plot_tree(filename, output=None, *, label_mode=None, layout=None, size=1000,
              higher_colors=None, subclade_colors=None, use_edge_lengths=True,
              edge_width=1.2, tip_size=6, arc_width=8, min_clade_size=2,
              strict_monophyly=False, title=None, radial_spacing=None,
              clade_size_power=None):
    """Read an .nwk file, save SVG, and return (canvas, axes, tree_mark).

    Defaults follow the two flags at the top of this file. Tip circles and pure
    subclade branches share the exact subclade shade. Branches joining multiple
    subclades within one highest group use its base color; mixed groups are gray.
    No leaf text is drawn. Scientific arc/legend names are italicized.

    size is the square tree panel in pixels; radial legends add canvas width.
    use_edge_lengths=False uses unit edges (aligned tips in circular mode).
    min_clade_size and strict_monophyly affect circular arcs only.
    Consecutive same-subclade arc fragments get one label at their combined
    midpoint. A leader is used only when that combined span is too short.

    radial_spacing="balanced" gives each subclade angular weight n**p, where
    n is its number of tips and p is clade_size_power (default 0.35). Lower p
    compresses common subclades relative to rare ones. p=0 gives equal total
    angular weight to each subclade; p=1 gives equal weight to every tip.
    radial_spacing="original" restores Toytree's unrooted daylight layout.
    These options only change angles, never topology or branch lengths.
    The balanced layout is root-dependent. Long branches and zero-length
    edges can still limit readability; use tip_size=3 for crowded plots.
    """
    mode = _mode(label_mode)
    layout = TREE_LAYOUT if layout is None else layout
    if layout not in ("radial", "circular"):
        raise ValueError('TREE_LAYOUT must be "radial" or "circular".')
    radial_spacing = RADIAL_SPACING if radial_spacing is None else radial_spacing
    clade_size_power = CLADE_SIZE_POWER if clade_size_power is None else clade_size_power
    if layout == "radial" and radial_spacing not in ("balanced", "original"):
        raise ValueError('radial_spacing must be "balanced" or "original".')
    if size < 400 or min_clade_size < 1 or min(edge_width, tip_size, arc_width) <= 0:
        raise ValueError("Use size >= 400 and positive sizes / widths.")
    tree = toytree.tree(str(filename))
    if tree.ntips < 2:
        raise ValueError("At least two tips are required.")
    higher, lower, bounds, counts = _summarize(tree, mode)
    keys = set(lower[:tree.ntips])
    higher_colors, generated = _palette(keys, mode, higher_colors)
    colors = generated if subclade_colors is None else dict(subclade_colors)
    missing = keys - colors.keys()
    if missing:
        raise ValueError(f"subclade_colors is missing: {sorted(missing)}")
    lengths = np.array([n.dist for n in tree if not n.is_root()])
    if use_edge_lengths:
        if not np.all(np.isfinite(lengths)) or np.any(lengths < 0):
            raise ValueError("Branch lengths must be finite and nonnegative.")
        if not np.any(lengths > 0):
            raise ValueError("All lengths are zero; use use_edge_lengths=False.")
    draw_tree = tree
    if not use_edge_lengths:
        draw_tree = tree.set_node_data("dist", default=1.0)
        if layout == "circular":
            draw_tree = draw_tree.mod.edges_extend_tips_to_align()

    rows = _legend_rows(keys, mode, higher_colors, colors) if layout == "radial" else []
    rows_per_column = max(1, int((size - 80) // 23))
    longest = max(len(_name(k, mode)) for k in keys)
    column_width = max(180, longest * 7 + 75)
    legend_width = int(np.ceil(len(rows) / rows_per_column)) * column_width
    margin = 40 if layout == "radial" else min(size * .27, max(65, longest * 5.8))
    canvas = toyplot.Canvas(width=size + legend_width, height=size)
    axes = canvas.cartesian(bounds=(margin, size - margin, margin, size - margin), padding=0)
    axes.show = False
    axes.aspect = "fit-range"
    edge_colors = [colors[sub] if sub is not None else higher_colors.get(parent, MIXED_COLOR)
                   for parent, sub in zip(higher, lower)]
    balanced = layout == "radial" and radial_spacing == "balanced"
    # Use a cheap provisional layout for the custom coordinates. This avoids
    # running Toytree's daylight optimization only to overwrite its result.
    initial_layout = "r" if balanced else ("unr" if layout == "radial" else "c")
    _, _, mark = draw_tree.draw(
        axes=axes, layout=initial_layout, edge_type="c",
        use_edge_lengths=True, tip_labels=False, tip_labels_align=False,
        node_labels=False, node_sizes=0, edge_colors=edge_colors,
        edge_widths=edge_width, scale_bar=False,
    )
    if balanced:
        mark.ntable, mark.radial_sectors = _balanced_radial_coordinates(
            draw_tree, lower[:tree.ntips], clade_size_power
        )
        mark.ttable = mark.ntable[:tree.ntips].copy()
        mark.layout = "unr"
    mark.tip_labels_align = False
    # Overlay explicitly: one circle at each actual tip, including unequal lengths.
    tip_colors = [colors[k] for k in lower[:tree.ntips]]
    mark.tip_circle_mark = axes.scatterplot(
        mark.ntable[:tree.ntips, 0], mark.ntable[:tree.ntips, 1],
        marker="o", size=tip_size, color=tip_colors, mstyle={"stroke": "none"},
    )
    if layout == "circular":
        center, limit = _arcs(axes, tree, mark, lower, bounds, counts, colors, mode,
                              size - 2 * margin, 1.34, arc_width,
                              min_clade_size, strict_monophyly)
    else:
        xy = np.asarray(mark.ntable)
        center = (xy.min(axis=0) + xy.max(axis=0)) / 2
        limit = max(np.ptp(xy, axis=0)) * .55
        if not np.isfinite(limit) or limit <= 0:
            raise ValueError("The tree has no positive drawing extent.")
        _draw_legends(canvas, rows, size, column_width, rows_per_column)
        mark.arc_label_stats = {"inside": 0, "outside": 0}
    axes.x.domain.min, axes.x.domain.max = center[0] - limit, center[0] + limit
    axes.y.domain.min, axes.y.domain.max = center[1] - limit, center[1] + limit
    if title:
        canvas.text(size / 2, 18, escape(title), style={"font-size": "13px"})
    output = Path(output) if output is not None else Path(f"{filename}.{mode}.{layout}.svg")
    toyplot.svg.render(canvas, str(output))
    return canvas, axes, mark

In [10]:
plot_tree(
    "nwk/E_esm3di.nwk",
    output="svg/E_esm3di.svg",
)
plot_tree(
    "nwk/E_prostt5.nwk",
    output="svg/E_prostt5.svg",
)
plot_tree(
    "nwk/E_af2.nwk",
    output="svg/E_af2.svg",
)

(<toyplot.canvas.Canvas at 0x7f2c3e2fefd0>,
 <toytree.drawing.src.mark_toytree.ToyTreeMark at 0x7f2c3e2ff4d0>)

### Filoviridae - based on scientific names (e.g. Orthomarburgvirus-marburgense_...)

In [11]:
# -------------------- CHANGE THESE FLAGS --------------------
LABEL_MODE = "scientific"  # "scientific" or "six_letter"
TREE_LAYOUT = "radial"    # "radial" (legends) or "circular" (arcs)

# Radial spacing only. These settings do not change branch lengths.
RADIAL_SPACING = "balanced"  # "balanced" or "original" (Toytree layout)
CLADE_SIZE_POWER = 0.2       # 0: equal space per subclade; 1: equal per tip
# Smaller values compress densely sampled subclades more strongly.
# ---------------------------------------------------------------

In [12]:
plot_tree(
    "/work/FAC/FBM/DBC/cdessim2/default/dkim5/projects/viral/ebola/nwk/gp_foldtree_esm3di.renamed.nwk",
    output="svg/gp_foldtree_esm3di.svg",
    tip_size=12,
)
plot_tree(
    "/work/FAC/FBM/DBC/cdessim2/default/dkim5/projects/viral/ebola/nwk/gp_foldtree_prostt5.renamed.nwk",
    output="svg/gp_foldtree_prostt5.svg",
    tip_size=12,
)

(<toyplot.canvas.Canvas at 0x7f2c3ee42ea0>,
 <toytree.drawing.src.mark_toytree.ToyTreeMark at 0x7f2c3e4f6850>)